# PGM Soil Profile Setup - DFS2 Generator

This notebook orchestrates generation of Task 4 DFS2 maps for soil-profile cell properties:
- Wilting point (water content, unit `-`)
- Field capacity (water content, unit `-`)

## Workflow Overview
1. **Setup Imports**: Resolve repository path and import module functions
2. **Configure Inputs**: Set user-editable source and output naming parameters
3. **Build Derived Paths**: Generate full input/output paths from configuration
4. **Parse Soil Profiles**: Read profile text files and build `profile_table.csv`
5. **Generate DFS2 Outputs**: Write `grid_codes.dfs2` and per-cell FC/WP maps
6. **Validate Outputs**: Save `summary.csv` and run diagnostic checks

## Step 1: Environment and Module Import

This cell configures notebook-relative paths and imports the `soil_profile_setup` orchestrator module from `src`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
SRC_DIR = REPO_ROOT.joinpath("src")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from plant_growth_module import soil_profile_setup

print("Repo root:   ", REPO_ROOT)

## Step 2A: User-Editable Parameters

Set only the high-level inputs you may want to change per run: source result folder, source DFS2 filename, one text-file glob pattern, and output naming.

In [ ]:
# User-editable parameters
RESULTS_DIR = REPO_ROOT.joinpath(
    r"sample_data\soil-profile-setup\Cernici16_Ben_v101_WM_AD.she - Result Files"
)
DFS2_GRIDCODES_IN = "Cernici16_Ben_v101_WM_AD_PreProcessed.DFS2"
PROFILE_GRID_ITEM_HINT = "Profile Grid Codes"
# Single glob pattern for soil profile text files.
SOIL_PROFILE_TXT_GLOB = "*PreProcessed_SoilProf*.txt"

OUTPUT_DIR = REPO_ROOT.joinpath(
    "output_data", "pgm_soil_profile_setup", "Cernici16_Ben_v101_WM_AD"
)
# Output naming parameters
OUTPUT_PREFIX_WP = "wilting_point"
OUTPUT_PREFIX_FC = "field_capacity"


print("Model results:           ", RESULTS_DIR)
print("Preprocessed filename:   ", DFS2_GRIDCODES_IN)
print("TXT pattern:             ", SOIL_PROFILE_TXT_GLOB)
print("Output dir:              ", OUTPUT_DIR)

## Step 2B: Derived Paths and Output Locations

This cell computes absolute input/output paths from the user-editable parameters and ensures output directories exist.

In [ ]:
# Generated paths derived from the user-editable parameters in the previous cell.

PREPROCESSED_DFS2 = RESULTS_DIR.joinpath(DFS2_GRIDCODES_IN)


WP_OUTPUT_DIR = OUTPUT_DIR.joinpath(OUTPUT_PREFIX_WP)
FC_OUTPUT_DIR = OUTPUT_DIR.joinpath(OUTPUT_PREFIX_FC)
PROFILE_TABLE_CSV_PATH = OUTPUT_DIR.joinpath("profile_table.csv")
SUMMARY_CSV_PATH = OUTPUT_DIR.joinpath("summary.csv")
GRID_CODES_DFS2_PATH = OUTPUT_DIR.joinpath("grid_codes.dfs2")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

matching_txt_files = sorted(RESULTS_DIR.glob(SOIL_PROFILE_TXT_GLOB))

print("Preprocessed DFS2:   ", PREPROCESSED_DFS2)
print("WP output dir:       ", WP_OUTPUT_DIR)
print("FC output dir:       ", FC_OUTPUT_DIR)
print("Matched txt files:   ", len(matching_txt_files))
for txt in matching_txt_files[:10]:
    print("  -", txt.name)

## Step 3: Parsing Configuration

The profile text format can vary between models. Parsing is handled in module code, while this notebook exposes manual overrides for project-specific fixes.

Supported parser behavior:
- Detect grid code from filename (e.g., `*SoilProf*.txt`, `*profile*.txt`, or other names with an embedded index)
- Extract soil name ranges by cell index
- Extract wilting point and field capacity from profile sections

If needed, configure manual overrides in the next cell.

In [ ]:
# Optional manual overrides if parsing needs corrections for a specific project.
# Format:
# MANUAL_CELL_RANGES_OVERRIDES[grid_code] = [(soil_name, start_cell, end_cell), ...]
# MANUAL_PROPERTY_OVERRIDES[grid_code] = {soil_name: (wilting_point, field_capacity), ...}
MANUAL_CELL_RANGES_OVERRIDES: dict[int, list[tuple[str, int, int]]] = {}
MANUAL_PROPERTY_OVERRIDES: dict[int, dict[str, tuple[float, float]]] = {}

print("Manual overrides loaded.")

## Step 4: Parse Soil Profile Text Files

Runs the parser over all matching soil-profile `*.txt` files, builds a normalized profile table, and saves it as `profile_table.csv`.

In [ ]:
parsed_by_grid, profile_table, max_cell_index = (
    soil_profile_setup.parse_soil_profile_texts(
        results_dir=RESULTS_DIR,
        soil_profile_txt_glob=SOIL_PROFILE_TXT_GLOB,
        profile_table_csv_path=PROFILE_TABLE_CSV_PATH,
        manual_cell_ranges_overrides=MANUAL_CELL_RANGES_OVERRIDES,
        manual_property_overrides=MANUAL_PROPERTY_OVERRIDES,
    )
)

print("Max soil cell index: ", max_cell_index)
print("Profile table CSV:", PROFILE_TABLE_CSV_PATH)
display(profile_table.head(100))

## Step 5: Generate DFS2 Maps

Uses parsed profile values plus profile grid codes from the preprocessed DFS2 to write:
- `grid_codes.dfs2`
- per-cell wilting point DFS2 maps
- per-cell field capacity DFS2 maps

In [ ]:
output_index = soil_profile_setup.generate_soil_property_dfs2_outputs(
    preprocessed_dfs2=PREPROCESSED_DFS2,
    profile_grid_item_hint=PROFILE_GRID_ITEM_HINT,
    parsed_by_grid=parsed_by_grid,
    max_cell_index=max_cell_index,
    wp_output_dir=WP_OUTPUT_DIR,
    fc_output_dir=FC_OUTPUT_DIR,
    output_prefix_wp=OUTPUT_PREFIX_WP,
    output_prefix_fc=OUTPUT_PREFIX_FC,
    grid_codes_dfs2_path=GRID_CODES_DFS2_PATH,
)

print(f"Generated {len(output_index) * 2} DFS2 files in {OUTPUT_DIR}")
print("Grid codes DFS2:", GRID_CODES_DFS2_PATH)
display(output_index.head(20))

## Step 6: Build Output Summary

Creates a compact summary table with generated file paths and existence flags, then saves it as `summary.csv`.

In [ ]:
summary = soil_profile_setup.build_soil_profile_summary(
    output_index=output_index,
    summary_csv_path=SUMMARY_CSV_PATH,
)
print("Summary CSV:", SUMMARY_CSV_PATH)
summary